In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.utils.data as tdata
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import random
import os
from waveform_model import WaveformPredictor

if torch.cuda.is_available():
  device = torch.device('cuda') 
elif torch.mps.is_available():
  device = torch.device('mps')
else:
  device = torch.device('cpu')

def set_seed(seed_value=42):
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        # torch.backends.cudnn.deterministic = True
        # torch.backends.cudnn.benchmark = False
        
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed_value)

set_seed(42)

In [ ]:
from gwosc.datasets import find_datasets
from gwosc import datasets
from gwosc.datasets import event_gps

print("List of available catalogs:")
print(find_datasets(type="catalog"))

In [ ]:
gwtc_events = datasets.find_datasets(type='events', catalog='GWTC-4.0')
print("Available events:", gwtc_events)

In [ ]:
from gwpy.timeseries import TimeSeries
import requests

event_name = 'GW240109_050431-v1'
gps = event_gps(event_name)
print("GPS time:", gps)

freq_resolution = 4 #Hz
window_size = 15
start = -15 # relative to gps central time
duration = 30
upscaling = 1e20
segment = (int(gps) + start, int(gps) + start + duration) #(int(gps) - window_size, int(gps) + window_size)

wave_data = TimeSeries.fetch_open_data("H1", *segment, verbose=True, cache=True)


r = requests.get(
  f"https://gwosc.org//eventapi/json/event/{event_name.replace('-','/')}/",
  )
param_data = r.json()['events'][event_name]

strain_seg = wave_data.value
fs = int(wave_data.sample_rate.value)
NFFT = int(fs / freq_resolution)

plot = wave_data.plot()

In [ ]:
param_data

In [ ]:
params_of_interest = ['chirp_mass_source','mass_1_source','mass_2_source','total_mass_source','final_mass_source','chi_eff','luminosity_distance','redshift'] # plus start to get the starting time

In [ ]:
class Dataset(tdata.Dataset):
    def __init__(self,wave_data,param_data,params_of_interest,start,duration,segment_splits,device,upscaling):
        super().__init__()

        self.device = device
        self.param_data = []
        self.waveform_data = []
        self.segment_size = wave_data.size // segment_splits
        self.segment_duration = duration / segment_splits
        self.upscaling = upscaling

        for s in range(segment_splits):
            self.param_data.append([param_data[p] for p in params_of_interest] + [start + s*self.segment_duration])
            self.waveform_data.append(np.array(wave_data[s*self.segment_size : (s+1)*self.segment_size]))

        self.param_data = torch.tensor(self.param_data).to(torch.float32).to(device)
        self.waveform_data = torch.tensor(self.waveform_data).unsqueeze(2).to(torch.float32).to(device) * upscaling

    def __getitem__(self, index):
        return self.param_data[index], self.waveform_data[index]
    
    def __len__(self):
        return len(self.param_data)


In [ ]:
from torch.utils.data import random_split

dataset = Dataset(wave_data=wave_data,param_data=param_data,params_of_interest=params_of_interest,start=start,duration=duration,segment_splits=480,device=device,upscaling=upscaling)
dlen = len(dataset)

train_dataset, valid_dataset = random_split(
    dataset = dataset, 
    lengths = [int(0.8*dlen), int(0.2*dlen)],
    generator = torch.Generator().manual_seed(42)
)

train_loader = tdata.DataLoader(train_dataset,batch_size=20)
valid_loader = tdata.DataLoader(valid_dataset,batch_size=10)

In [ ]:
model_parameters = {
    'device': device,
    'pad_token': float(0.0),
    'max_seq_len': int(512),
    'wave_dim': int(1),
    'input_size': int(9),
    'encoder_hid_dim': int(32),
    'embed_dim': int(128),
    'encoder_dropout': float(0.3),
    'decoder_dropout': float(0.3),
    'decoders_num': int(4),
    'masked_heads': int(8),
    'ff_hid_dim': int(512),
    'decoder_hid_dim': int(32),
    'wave_decoder_dropout': float(0.3),
    }


waveform_predictor = WaveformPredictor(model_parameters).to(torch.float32).to(device)

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(waveform_predictor.parameters(),lr=1e-3)

In [ ]:
train_loss = []
valid_loss = []

for epoch in tqdm(range(200)):
    waveform_predictor.train()
    tloss = 0

    for x,y in train_loader:
        py = waveform_predictor(x,y)[:,1:]

        optimizer.zero_grad()
        loss = criterion(py,y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(waveform_predictor.parameters(), max_norm=1.0)
        optimizer.step()

        tloss += loss.item()
    train_loss.append(tloss)

    waveform_predictor.eval()
    tloss = 0

    for x,y in valid_loader:
        py = waveform_predictor(x,y)[:,1:]
        loss = criterion(py,y)
        tloss += loss.item()
    valid_loss.append(tloss)
    
        

In [ ]:
plt.plot(train_loss,label='Train')
plt.plot(valid_loss,label='Validation')
plt.legend()
plt.show()

In [ ]:
def inference(input,time_steps):
    input = input.unsqueeze(0)
    input = input.unsqueeze(0)
    window = waveform_predictor.parameters_encoder(input)
    output = []

    for t in tqdm(range(time_steps)):
        pwave = waveform_predictor.decode(window)[:,-1:,:]
        window = torch.cat([window, pwave], dim=1)
        if window.shape[1] > model_parameters['max_seq_len']:
            window = window[:,1:,:]

        pwave = waveform_predictor.wave_decoder(pwave[:,-1:,:])[0][0][0].detach().cpu()
        output.append(pwave)
    
    return output

In [ ]:
with torch.no_grad():
    output = inference(dataset[0][0],100)

In [ ]:
plt.plot(dataset[0][1][:100].cpu(),label='Target')
plt.legend()
plt.show()
plt.plot(output,label='Predicted')
plt.legend()
plt.show()